# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.tooling import tool          # turns a function into something the model can call
from lib.llm import LLM               # thin OpenAI wrapper, used for the judge
from lib.parsers import PydanticOutputParser   # JSON string -> Pydantic object

In [3]:
# .env lives at the repo root, two folders up from project/starter.
load_dotenv(dotenv_path="../../.env", override=True)

# Fail here, loudly, rather than three layers deep inside a tool call.
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.getenv(key), f"{key} is missing from .env"

# Tavily needs its key handed over explicitly.
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# The OpenAI key is deliberately NOT stored in a variable. The OpenAI SDK reads
# OPENAI_API_KEY and OPENAI_BASE_URL from the environment on its own, and Chroma's
# embedding function is built on that same SDK, so both inherit the gateway.

print("environment loaded")

environment loaded


In [4]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(name="udaplay", embedding_function=embedding_fn)

print(f"collection ready — {collection.count()} documents")

collection ready — 15 documents


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds most relevant results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)
    return results["documents"][0]

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Explanation of why the documents are or are not useful")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")

    prompt = (
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}"
    )

    response = llm.invoke(input=prompt, response_format=EvaluationReport)

    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return {"useful": report.useful, "description": report.description}

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> list:
    """Web search: Finds results on the internet when the vector DB has no useful answer.
    args:
    - question: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - title: title of the web page
    - url: source URL
    - content: relevant excerpt from the page
    """
    tavily = TavilyClient(api_key=TAVILY_API_KEY)
    response = tavily.search(query=question)
    return response["results"]

In [8]:
# --- smoke test: retrieve_game ---
docs = retrieve_game.func("When was Pokémon Gold and Silver released?")
for d in docs:
    print(d)
    print("---")

[Game Boy Color] Pokémon Gold and Silver (1999) - Role-playing, published by Nintendo. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
---
[Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Role-playing, published by Nintendo. Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.
---
[Nintendo 64] Super Mario 64 (1996) - Platformer, published by Nintendo. A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
---


In [9]:
# --- smoke test: evaluate_retrieval ---
# Case 1: good retrieval — Pokémon question, relevant docs
good_docs = retrieve_game.func("When was Pokémon Gold and Silver released?")
report_good = evaluate_retrieval.func(
    question="When was Pokémon Gold and Silver released?",
    retrieved_docs=good_docs
)
print("Case 1 (should be useful=True):")
print(report_good)

print()

# Case 2: bad retrieval — Mortal Kombat X not in corpus, noisy docs
bad_docs = retrieve_game.func("Was Mortal Kombat X released for PlayStation 5?")
report_bad = evaluate_retrieval.func(
    question="Was Mortal Kombat X released for PlayStation 5?",
    retrieved_docs=bad_docs
)
print("Case 2 (should be useful=False):")
print(report_bad)

Case 1 (should be useful=True):
{'useful': True, 'description': 'The retrieved documents contain relevant information about the release of Pokémon Gold and Silver, specifically stating that they were released in 1999. Although the other documents discuss different Pokémon games, they do not detract from the usefulness of the first document, which directly answers the question. Therefore, the documents are sufficient to respond to the query.'}

Case 2 (should be useful=False):
{'useful': False, 'description': "The retrieved documents do not contain any information regarding Mortal Kombat X or its release on PlayStation 5. Instead, they focus on other games, specifically Marvel's Spider-Man and Gran Turismo 5, which are unrelated to the query. Therefore, these documents are not useful for answering whether Mortal Kombat X was released for PlayStation 5."}


In [10]:
# --- smoke test: game_web_search ---
from tavily import TavilyClient
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

web_results = game_web_search.func(question="Was Mortal Kombat X released for PlayStation 5?")
for r in web_results[:2]:
    print(r["title"])
    print(r["url"])
    print(r["content"][:200])
    print("---")

Mortal Kombat X PS5 Gameplay Review
https://www.youtube.com/watch?v=-JPXripEMoA
It's Mortal Kombat X for on PlayStation 5 taking a look at this entry in the long-running fighting series. So, what we're getting here is a 1080p 60 FPS. It features a large roster of characters with 
---
Mortal Kombat X - PS5 Gameplay
https://www.youtube.com/watch?v=tqsw711ZuAk
Mortal Kombat X is a 2015 fighting game. Entertainment for Microsoft Windows, PlayStation 4, and Xbox One.
---


### Agent

In [11]:
from lib.agents import Agent

INSTRUCTIONS = """You are UdaPlay, a research assistant for the video game industry.

Follow this procedure for every question about a game:

1. Call retrieve_game first, with the user's question as the query.
2. Call evaluate_retrieval, passing the original question and the documents
   you just retrieved.
3. If the evaluation returns useful=true, answer from those documents alone.
   If it returns useful=false, call game_web_search and answer from those
   results instead.

Never call game_web_search before steps 1 and 2. The internal database is
the preferred source.

When you answer:
- State the platform and year when the question is about a release.
- Distinguish the platforms a game was RELEASED for from platforms where it
  is merely playable through backwards compatibility. These are not the same
  claim, and web results often blur them.
- If you used web results, name the source URLs you relied on.
- If neither source answers the question, say so plainly. Do not guess."""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

print("agent ready with tools:", [t.name for t in agent.tools])

agent ready with tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']


In [12]:
from lib.messages import AIMessage


def tools_used(run, this_turn_only=True):
    """Read back which tools actually ran, from the state the machine recorded.

    In a session, the message list a run starts with is the previous turn's whole
    conversation. Counting all of it would report that turn's tools as well, so by
    default we skip what came in and count only what this run added.
    """
    messages = run.get_final_state()["messages"]
    if this_turn_only:
        carried_in = len(run.snapshots[0].state_data["messages"])
        messages = messages[carried_in:]

    names = []
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            names.extend(call.function.name for call in msg.tool_calls)
    return names


QUERIES = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

# A separate session per query, so no query can answer from the previous one's
# conversation. Carrying context across turns is tested on purpose later.
for i, question in enumerate(QUERIES, start=1):
    run = agent.invoke(question, session_id=f"q{i}")
    state = run.get_final_state()

    print(f"Q{i}: {question}")
    print(f"tools fired : {tools_used(run)}")
    print(f"steps       : {len(run.snapshots)}")
    print(f"tokens      : {state.get('total_tokens', 0)}")
    print()
    print(state["messages"][-1].content)
    print("=" * 78)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q1: When was Pokémon Gold and Silver released?
tools fired : ['retrieve_game', 'evaluate_retrieval']
steps       : 7
tokens      : 2201

Pokémon Gold and Silver were released for the Game Boy Color in 1999.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q2: Which one was the first 3D platformer Mario game?
tools fired : ['retrieve_game', 'evaluate_r

### Multiple queries in one session

A follow-up question using "it" only makes sense if the agent still has the previous turn.
The proof is the contrast: the same follow-up is asked twice, once in a session that has
history and once in a session that has none.

In [13]:
import contextlib
import io

# Safe to re-run: create each session if it is new, then empty it.
for name in ("chat", "cold"):
    agent.memory.create_session(name)
    agent.reset_session(name)


def ask(question, session_id):
    """Invoke the agent with the step trace suppressed — it is shown in the cell above."""
    with contextlib.redirect_stdout(io.StringIO()):
        return agent.invoke(question, session_id=session_id)


def report(label, run):
    state = run.get_final_state()
    print(label)
    print(f"  answer   : {state['messages'][-1].content}")
    print(f"  tools    : {tools_used(run)}")
    print(f"  messages : came in with {len(run.snapshots[0].state_data['messages'])}, "
          f"ended with {len(state['messages'])}")
    print(f"  tokens   : {state.get('total_tokens', 0)}")
    print()


FOLLOW_UP = "Who published it?"

report("chat, turn 1 — establishes the subject",
       ask("When was Pokémon Gold and Silver released?", "chat"))

report("chat, turn 2 — 'it' must come from turn 1",
       ask(FOLLOW_UP, "chat"))

report("cold, turn 1 — same words, no history to resolve 'it'",
       ask(FOLLOW_UP, "cold"))

print("runs kept in 'chat':", len(agent.get_session_runs("chat")))
print("runs kept in 'cold':", len(agent.get_session_runs("cold")))

chat, turn 1 — establishes the subject
  answer   : Pokémon Gold and Silver were released for the Game Boy Color in 1999.
  tools    : ['retrieve_game', 'evaluate_retrieval']
  messages : came in with 0, ended with 7
  tokens   : 2201

chat, turn 2 — 'it' must come from turn 1
  answer   : Pokémon Gold and Silver were published by Nintendo.
  tools    : ['retrieve_game', 'evaluate_retrieval']
  messages : came in with 7, ended with 13
  tokens   : 3341

cold, turn 1 — same words, no history to resolve 'it'
  answer   : The retrieved documents provide information about the publishers of several games:

1. **Minecraft (2014)** - Published by **Mojang Studios** for Xbox One.
2. **Marvel's Spider-Man (2018)** - Published by **Sony Interactive Entertainment** for PlayStation 4.
3. **Pokémon Gold and Silver (1999)** - Published by **Nintendo** for Game Boy Color.

If you have a specific game in mind, please let me know!
  tools    : ['retrieve_game', 'evaluate_retrieval']
  messages : came i

### (Optional) Advanced

In [14]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

#### Long-term memory

`ShortTermMemory` holds conversations in a Python dictionary, so everything is lost when the
kernel stops. Long-term memory writes facts to disk instead, in a second collection beside the
game corpus, so a fact learned from the web is still there after a restart.

Three defects in `lib/` had to be fixed before this could work at all — see `BUGS.md` B2, B5
and B8.

In [15]:
from lib.vector_db import VectorStoreManager
from lib.memory import LongTermMemory, MemoryFragment

MEMORY_OWNER = "udaplay"
MEMORY_NAMESPACE = "games"

# Same folder as the game corpus, so "udaplay" and "long_term_memory" sit side by side.
store_manager = VectorStoreManager(os.getenv("OPENAI_API_KEY"), persist_path="chromadb")

# reset=False keeps whatever is already stored; reset=True would delete it.
ltm = LongTermMemory(store_manager, reset=False)

print("collections on disk :", [c.name for c in store_manager.chroma_client.list_collections()])
print("embedding model     :", store_manager.embedding_function.model_name)
print("facts remembered    :", ltm.vector_store._collection.count())

collections on disk : ['udaplay', 'long_term_memory']
embedding model     : text-embedding-3-small
facts remembered    : 1


In [16]:
# Same tool name, upgraded: it now searches remembered facts as well as the game corpus.
# Redefined here on purpose — the canonical queries further up ran against the corpus
# alone, which is what made the Mortal Kombat X fallback fire.
@tool
def retrieve_game(query: str) -> list[str]:
    """Semantic search: Finds most relevant results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game

    Some results may be facts remembered from earlier web searches, marked [remembered].
    """
    from_corpus = collection.query(query_texts=[query], n_results=3)["documents"][0]

    recalled = ltm.search(
        query_text=query,
        owner=MEMORY_OWNER,
        namespace=MEMORY_NAMESPACE,
        limit=2,
    )
    from_memory = [f"[remembered] {fragment.content}" for fragment in recalled.fragments]

    return from_corpus + from_memory

In [17]:
# Rebuilt so it picks up the memory-aware retrieve_game. Same instructions as before.
memory_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)


def ask_and_remember(question, session_id):
    """Answer the question, and store the answer if it took a web search to get it.

    The write-back is decided here rather than by the agent: if game_web_search is in
    the trace, the corpus could not answer, so the answer is worth keeping. What gets
    stored is the agent's own answer, not the raw web results — the answer is already
    the distilled version.
    """
    run = memory_agent.invoke(question, session_id=session_id)
    state = run.get_final_state()
    answer = state["messages"][-1].content
    used = tools_used(run)

    if "game_web_search" in used:
        ltm.register(MemoryFragment(
            content=f"{question}\n{answer}",
            owner=MEMORY_OWNER,
            namespace=MEMORY_NAMESPACE,
        ))

    print(f"question : {question}")
    print(f"tools    : {used}")
    print(f"stored   : {'yes — came from the web' if 'game_web_search' in used else 'no'}")
    print(f"remembered facts now: {ltm.vector_store._collection.count()}")
    print()
    print(answer)
    return run

In [18]:
# Start from an empty memory so the demonstration is honest. reset=True is the
# destructive path from the B5 fix, used deliberately here rather than by accident.
ltm = LongTermMemory(store_manager, reset=True)
print("facts remembered at start:", ltm.vector_store._collection.count())
print()

QUESTION = "Was Mortal Kombat X released for PlayStation 5?"

print("=" * 78)
print("FIRST ASK — nothing in memory, the corpus does not know this game")
print("=" * 78)
_ = ask_and_remember(QUESTION, session_id="memory-learn")

print()
print("=" * 78)
print("SECOND ASK — different session, so the only thing carried over is memory")
print("=" * 78)
_ = ask_and_remember(QUESTION, session_id="memory-recall")

facts remembered at start: 0

FIRST ASK — nothing in memory, the corpus does not know this game
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
question : Was Mortal Kombat X released for PlayStation 5?
tools    : ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
stored   : yes — came from the web
remembered facts now: 1

Mortal Kombat X was not released specifically for PlayStation 5. It was originally released for PlayStation 4 on April 14, 2015. However, it is playable on PlayStation 5 through backwards compatibility, meaning you can play the PS4 version on the PS5, but it was not released a

In [19]:
# --- the restart test ---
# Run this after restarting the kernel and re-running every cell above EXCEPT the one
# that resets memory. Nothing from the previous kernel is in scope any more: the agent
# is new, ShortTermMemory is empty, and no session history exists. The only thing that
# crossed the restart is what is written in chromadb/ on disk.
QUESTION = "Was Mortal Kombat X released for PlayStation 5?"

print("facts that survived the restart:", ltm.vector_store._collection.count())
print()
_ = ask_and_remember(QUESTION, session_id="after-restart")

facts that survived the restart: 1

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
question : Was Mortal Kombat X released for PlayStation 5?
tools    : ['retrieve_game', 'evaluate_retrieval']
stored   : no
remembered facts now: 1

Mortal Kombat X was not released specifically for PlayStation 5. It was originally released for PlayStation 4 on April 14, 2015. However, it is playable on PlayStation 5 through backwards compatibility, meaning you can play the PS4 version on the PS5, but it was not released as a native PS5 title. 

For more details, you can refer to the PlayStation website [here](https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved) and the Wikipedia page on Mortal 

### Evaluation

Three levels of evaluation, each catching different failures:

| Evaluator | Question | Input |
|---|---|---|
| `evaluate_final_response` | Was the answer right? | Agent's text output, judged by a second LLM |
| `evaluate_single_step` | Was the right tool picked? | One `AIMessage`'s tool calls |
| `evaluate_trajectory` | Was the path efficient? | All `Run.snapshots` — steps, tokens, tools |

A `TestCase` defines what each run is expected to do: the query, which tools must fire, an optional reference answer, and a step ceiling.

In [20]:
from lib.evaluation import AgentEvaluator, TestCase

evaluator = AgentEvaluator()

tc_pokemon = TestCase(
    id="tc-01",
    description="Retrieve a release date from the corpus",
    user_query="When was Pokémon Gold and Silver released?",
    expected_tools=["retrieve_game", "evaluate_retrieval"],
    reference_answer="Pokémon Gold and Silver were released in 1999 in Japan and 2000 in North America.",
    max_steps=8,
)

tc_mario = TestCase(
    id="tc-02",
    description="Reason across multiple corpus entries",
    user_query="Which one was the first 3D platformer Mario game?",
    expected_tools=["retrieve_game", "evaluate_retrieval"],
    reference_answer="Super Mario 64 was the first 3D platformer Mario game, released in 1996.",
    max_steps=8,
)

tc_mkx = TestCase(
    id="tc-03",
    description="Trigger the web-search fallback for a game not in the corpus",
    user_query="Was Mortal Kombat X released for PlayStation 5?",
    expected_tools=["retrieve_game", "evaluate_retrieval", "game_web_search"],
    reference_answer="No. Mortal Kombat X was released in 2015 for PS4, Xbox One, and PC — before the PS5 existed.",
    max_steps=10,
)

print("test cases ready:", [tc.id for tc in [tc_pokemon, tc_mario, tc_mkx]])

test cases ready: ['tc-01', 'tc-02', 'tc-03']


In [ ]:
import time

results = []

for tc in [tc_pokemon, tc_mario, tc_mkx]:
    t0 = time.time()
    run = agent.invoke(tc.user_query, session_id=f"eval-{tc.id}")
    elapsed = time.time() - t0

    state = run.get_final_state()
    answer = state["messages"][-1].content
    tokens = state.get("total_tokens", 0)

    r_final    = evaluator.evaluate_final_response(tc, answer, elapsed, tokens)
    r_step     = evaluator.evaluate_single_step(state["messages"], tc.expected_tools)
    r_traj     = evaluator.evaluate_trajectory(tc, run)

    results.append({
        "id":        tc.id,
        "final":     round(r_final.overall_score, 2),
        "step":      round(r_step.overall_score, 2),
        "traj":      round(r_traj.overall_score, 2),
        "steps":     r_traj.task_completion.steps_taken,
        "tokens":    tokens,
        "cost_usd":  round(r_traj.system_metrics.cost_estimate, 5),
        "feedback":  r_final.feedback,
    })

print(f"{'id':<8} {'final':>6} {'step':>6} {'traj':>6} {'steps':>6} {'tokens':>7} {'cost':>9}")
print("-" * 56)
for r in results:
    print(f"{r['id']:<8} {r['final']:>6} {r['step']:>6} {r['traj']:>6} {r['steps']:>6} {r['tokens']:>7} {r['cost_usd']:>9}")

print()
print("feedback (final-response judge):")
for r in results:
    print(f"  {r['id']}: {r['feedback']}")

In [ ]:
# --- degradation test ---
# Swap evaluate_retrieval's docstring for something vague, rebuild the agent,
# re-run tc_mkx, and observe the score. Then restore and confirm it recovers.
#
# This proves the eval measures real behaviour — not just that the agent ran.

@tool
def evaluate_retrieval_degraded(question: str, retrieved_docs: list[str]) -> dict:
    """Checks documents."""  # vague: the model no longer understands what to do or when
    llm = LLM(model="gpt-4o-mini")
    prompt = (
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}"
    )
    response = llm.invoke(input=prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)
    return {"useful": report.useful, "description": report.description}


degraded_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval_degraded, game_web_search],
    temperature=0.0,
)

t0 = time.time()
run_d = degraded_agent.invoke(tc_mkx.user_query, session_id="eval-degraded")
elapsed_d = time.time() - t0

state_d = run_d.get_final_state()
answer_d = state_d["messages"][-1].content
tokens_d = state_d.get("total_tokens", 0)

r_final_d = evaluator.evaluate_final_response(tc_mkx, answer_d, elapsed_d, tokens_d)
r_traj_d  = evaluator.evaluate_trajectory(tc_mkx, run_d)

print("degraded agent — tc-03 (Mortal Kombat X)")
print(f"  tools fired : {tools_used(run_d)}")
print(f"  final score : {round(r_final_d.overall_score, 2)}")
print(f"  traj  score : {round(r_traj_d.overall_score, 2)}")
print(f"  feedback    : {r_final_d.feedback}")
print()
print("baseline scores for comparison — final:", results[2]["final"], "  traj:", results[2]["traj"])